# Code mẫu — Regularization cho mạng neural

Mỗi kỹ thuật ở đây can thiệp vào một chỗ khác nhau. Dropout tác động lên activation, BatchNorm giữ lại thống kê của dữ liệu, còn early stopping chọn thời điểm dừng dựa trên validation. Ta xét riêng từng cơ chế, đổi đúng một điều kiện mỗi lần, để hiểu vì sao cùng đầu vào hay cùng mạng lại có thể cho kết quả khác nhau.

## Ví dụ 1: Dropout giữ đúng trung bình, không giữ từng đầu ra

Một vector activation có tám phần tử. Dropout với xác suất 0.5 sẽ biến mỗi phần tử thành 0, hoặc nhân nó lên 2. Ta dùng một mask đặt bằng tay để kiểm hệ số nhân. Sau đó chạy dropout thật hai lần, để xem mask có được rút lại không, kể cả khi đang ở trong `no_grad`.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(53)
h = torch.arange(1., 9.)
mask = torch.tensor([1., 0., 1., 0., 1., 0., 1., 0.])
print("mask tính tay:", (h * mask / .5).tolist())
dropout = nn.Dropout(p=.5)
dropout.train()
with torch.no_grad():
    first, second = dropout(h), dropout(h)
print("train lượt 1:", first.tolist())
print("train lượt 2:", second.tolist())
print("no_grad làm hai lượt bằng nhau:", torch.equal(first, second))
dropout.eval()
print("eval:", dropout(h).tolist())
assert torch.equal(dropout(h), h)
assert ((first == 0) | (first == 2 * h)).all()

mask tính tay: [2.0, 0.0, 6.0, 0.0, 10.0, 0.0, 14.0, 0.0]
train lượt 1: [0.0, 4.0, 0.0, 8.0, 0.0, 12.0, 0.0, 0.0]
train lượt 2: [0.0, 4.0, 6.0, 8.0, 0.0, 0.0, 14.0, 16.0]
no_grad làm hai lượt bằng nhau: False
eval: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]


Lấy activation 3 làm ví dụ. Sau dropout, nó thành 0 hoặc 6, mỗi khả năng một nửa, nên trung bình vẫn là $0.5\cdot0+0.5\cdot6=3$. Nhờ nhân 2, trung bình được giữ nguyên, dù từng lần chạy cho ra một vector khác.

Hai lượt train cho hai mask khác nhau, dù cả hai đều nằm trong `no_grad`. `no_grad` chỉ tắt việc ghi graph, không tắt dropout. Phải gọi `eval()` thì dropout mới trả lại nguyên vector. Lượt 1 bỏ năm phần tử, lượt 2 bỏ ba, chứ không phải lúc nào cũng đúng một nửa. Xác suất 0.5 là cho từng phần tử, không phải tỉ lệ cố định của cả vector.

## Ví dụ 2: BatchNorm, và một lượt validation chạy nhầm chế độ

Một batch gồm hai mẫu `(1, 10)` và `(3, 14)`. Mean theo từng cột là `(2, 12)`. Phương sai, khi chia cho kích thước batch, là `(1, 4)`. Ta tắt phần scale và shift học được bằng `affine=False`, và đặt momentum bằng 1, để thấy thẳng phép chuẩn hóa và các thống kê được ghi lại.

In [ ]:
import copy
import torch
from torch import nn

torch.set_num_threads(1)
batch = torch.tensor([[1., 10.], [3., 14.]])
bn = nn.BatchNorm1d(2, affine=False, momentum=1.)
bn.train()
train_output = bn(batch)
state = copy.deepcopy(bn.state_dict())
bn.eval()
eval_output = bn(batch)
rounded = lambda t: [[round(x, 4) for x in row] for row in t.tolist()]
print("train:", rounded(train_output))
print("running mean:", bn.running_mean.tolist())
print("running variance:", bn.running_var.tolist())
print("eval:", rounded(eval_output))
assert torch.allclose(bn.running_mean, torch.tensor([2., 12.]))
assert torch.allclose(bn.running_var, torch.tensor([2., 8.]))

train: [[-1.0, -1.0], [1.0, 1.0]]
running mean: [2.0, 12.0]
running variance: [2.0, 8.0]
eval: [[-0.7071, -0.7071], [0.7071, 0.7071]]


Ở chế độ train, BatchNorm chuẩn hóa bằng thống kê của chính batch, nên đầu ra gần ±1. Nó đồng thời ghi lại running mean và running variance để dùng về sau. Running variance là `(2, 8)`, chứ không phải `(1, 4)`, vì khi ghi lại, PyTorch chia cho $B-1$ thay vì $B$. Ở chế độ eval, BatchNorm dùng các con số đã ghi ấy, nên đầu ra chỉ còn ±0.7071. Cùng một batch, hai chế độ cho hai kết quả khác nhau.

Giờ thử một batch validation, dịch lên 100 so với batch train. Ta chạy nó hai lần, một lần ở chế độ train, một lần ở chế độ eval, rồi xem running mean có bị đổi không.

In [ ]:
validation = batch + 100
bn.train()
with torch.no_grad():
    bn(validation)
print("running mean sau validation ở train:", bn.running_mean.tolist())

bn.load_state_dict(state)
bn.eval()
before = bn.running_mean.clone()
with torch.no_grad():
    bn(validation)
print("running mean sau validation ở eval:", bn.running_mean.tolist())
print("buffer giữ nguyên:", torch.equal(before, bn.running_mean))
assert torch.equal(before, bn.running_mean)

running mean sau validation ở train: [102.0, 112.0]
running mean sau validation ở eval: [2.0, 12.0]
buffer giữ nguyên: True


Quên chuyển sang eval, running mean bị ghi đè thành 102 và 112, dù đang ở trong `no_grad`. Từ đó, model sẽ chuẩn hóa mọi dữ liệu theo thống kê của validation. Chuyển sang eval thì running mean giữ nguyên 2 và 12. Các thống kê này là một phần của model, nên checkpoint phải lưu chúng cùng với trọng số. `state_dict()` đã tự làm việc đó.

## Ví dụ 3: Chọn epoch tốt nhất trước, rồi mới so weight decay

Mười sáu điểm train nằm quanh đường $y=x^2$, có thêm nhiễu. Validation gồm 64 điểm nằm đúng trên đường sạch, để ta thấy rõ cơ chế. Ta train hai lần với cùng kiến trúc, cùng dữ liệu, cùng khởi tạo, chỉ đổi weight decay của AdamW. Early stopping dừng khi validation MSE không giảm thêm ít nhất $10^{-5}$ trong 15 epoch liền.

In [ ]:
import copy
import torch
from torch import nn

torch.set_num_threads(1)
generator = torch.Generator().manual_seed(9)
train_x = torch.linspace(-1, 1, 16).unsqueeze(1)
train_y = train_x.square() + .18 * torch.randn(16, 1, generator=generator)
valid_x = torch.linspace(-1, 1, 64).unsqueeze(1)
valid_y = valid_x.square()
print("shape train / validation:", tuple(train_x.shape), tuple(valid_x.shape))
print("MSE của đường sạch trên nhãn train có nhiễu:",
      round((train_x.square() - train_y).square().mean().item(), 5))

shape train / validation: (16, 1) (64, 1)
MSE của đường sạch trên nhãn train có nhiễu: 0.04911


Ngay cả đường đúng $x^2$ cũng có MSE 0.049 trên nhãn train, vì nhãn có nhiễu. Đây là một mốc để so. Nếu train MSE xuống thấp hơn mức này, model đang bắt đầu học thuộc cả nhiễu.

In [ ]:
def fit(decay):
    torch.manual_seed(29)  # Khởi tạo y hệt nhau cho mỗi giá trị decay.
    model = nn.Sequential(nn.Linear(1, 16), nn.Tanh(),
                          nn.Linear(16, 16), nn.Tanh(), nn.Linear(16, 1))
    optimizer = torch.optim.AdamW(model.parameters(), lr=.02, weight_decay=decay)
    best, best_epoch, waiting = float("inf"), 0, 0
    best_state, history = None, []
    for epoch in range(1, 121):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = (model(train_x) - train_y).square().mean()
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            train_mse = (model(train_x) - train_y).square().mean().item()
            valid_mse = (model(valid_x) - valid_y).square().mean().item()
        history.append((epoch, train_mse, valid_mse))
        if valid_mse < best - 1e-5:
            best, best_epoch, waiting = valid_mse, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            waiting += 1
        if waiting == 15:
            break
    last = history[-1]
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        restored_mse = (model(valid_x) - valid_y).square().mean().item()
    assert abs(restored_mse - best) < 1e-7
    return history, best_epoch, restored_mse, last

for decay in [0., .1]:
    history, best_epoch, restored, last = fit(decay)
    print("decay", decay, "(epoch, train MSE, valid MSE):")
    for record in [history[0], history[best_epoch - 1], last]:
        print(record[0], round(record[1], 5), round(record[2], 5))
    print("khôi phục epoch", best_epoch, "valid MSE", round(restored, 5))

decay 0.0 (epoch, train MSE, valid MSE):
1 0.1935 0.10965
26 0.04023 0.00367
41 0.03886 0.00852
khôi phục epoch 26 valid MSE 0.00367
decay 0.1 (epoch, train MSE, valid MSE):
1 0.19374 0.10982
26 0.04192 0.00364
41 0.03844 0.00791
khôi phục epoch 26 valid MSE 0.00364


Mỗi lượt in ba dòng: epoch đầu, epoch tốt nhất, và epoch lúc dừng. Ở cả hai lượt, epoch 26 cho validation MSE thấp nhất, khoảng 0.0036. Sau đó train MSE tiếp tục giảm xuống dưới mốc 0.049, còn validation MSE tăng gấp đôi. Model đang học nhiễu. Sau 15 epoch không cải thiện, vòng lặp dừng ở epoch 41.

Patience chỉ quyết định chờ bao lâu trước khi dừng. Model đem dùng phải là bản đã lưu ở epoch 26, chứ không phải bản ở epoch 41. Dòng assert kiểm đúng điều đó. Ngưỡng $10^{-5}$ yêu cầu mỗi lần cải thiện phải đủ lớn, nên epoch được giữ có thể khác một chút so với epoch có MSE nhỏ nhất tuyệt đối.

So hai lượt, decay 0.1 chỉ nhỉnh hơn rất ít ở epoch tốt nhất. Validation sạch là một điều kiện dựng ra để quan sát. Con số này không chứng minh decay 0.1 sẽ tốt cho bài khác. Validation ở đây đã được dùng để chọn cả epoch lẫn decay, nên muốn có đánh giá cuối cùng, cần một tập chưa tham gia vào lựa chọn nào.